# 9. Segmentare due documenti nella stessa pagina

Classifica e separa due documenti collocati nella stessa pagina.

## 1. Import e configurazione

Carica librerie, variabili di ambiente, schema e documento.

In [1]:
import json
import os
from pathlib import Path
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

In [2]:
load_dotenv(r"..\.env", override=True)
document_bytes = Path(r"..\file\esercizio9.pdf").read_bytes()
definizione = json.loads(
    Path(r"..\modelli\segmentazione_in_pagina.json").read_text(encoding="utf-8")
)

## 2. Classificazione e segmentazione

Crea l'analyzer e rileva i segmenti interni alla pagina.

In [3]:
with DefaultAzureCredential() as credential, ContentUnderstandingClient(
    endpoint=os.environ["FOUNDRY_SERVICES_ENDPOINT"], credential=credential,
) as client:
    client.begin_create_analyzer(
        analyzer_id="segmentazione_in_pagina",
        resource=ContentAnalyzer(definizione),
        allow_replace=True,
    ).result()
    try:
        result = client.begin_analyze_binary(
            analyzer_id="segmentazione_in_pagina",
            binary_input=document_bytes,
        ).result()
    finally:
        client.delete_analyzer(analyzer_id="segmentazione_in_pagina")

## 3. Segmenti

Mostra categoria, confidence e area occupata da ogni segmento.

In [4]:
segmenti = [
    {
        "id": segment["segmentId"],
        "categoria": segment["category"],
        "pagina_iniziale": segment["startPageNumber"],
        "pagina_finale": segment["endPageNumber"],
        "confidence": segment["confidence"],
        "source": segment["source"],
        "span": segment["span"],
    }
    for content in result.as_dict()["contents"]
    for segment in content["segments"]
]
print(json.dumps(segmenti, ensure_ascii=False, indent=2))

[
  {
    "id": "segment1",
    "categoria": "expense_report",
    "pagina_iniziale": 1,
    "pagina_finale": 1,
    "confidence": 0.99,
    "source": "D(1,0.77,0.69,3.73,0.69,3.73,2.59,0.77,2.59)",
    "span": {
      "offset": 0,
      "length": 231
    }
  },
  {
    "id": "segment2",
    "categoria": "restaurant_receipt",
    "pagina_iniziale": 1,
    "pagina_finale": 1,
    "confidence": 0.99,
    "source": "D(1,0.77,5.83,3.62,5.83,3.62,7.95,0.77,7.95)",
    "span": {
      "offset": 231,
      "length": 372
    }
  }
]
